In [1]:
from datetime import UTC, datetime

import pandas as pd
from aws import AthenaDataSource
from core import (
    BacktestTaskDefinition,
    CurrencyPair,
    EventBus,
    InMemoryTaskRepository,
    LogLevel,
    RecordingEventHandler,
    StrategyEvent,
    StrategyParameters,
    TaskManager,
    TickGranularity,
    TickGranularityFilter,
    configure_logging,
)
from snowball import SnowballStrategy

configure_logging(level=LogLevel.WARNING)

<Logger core (WARNING)>

In [ ]:
INSTRUMENT = CurrencyPair.of("USD_JPY")
START_AT = datetime(2026, 1, 1, tzinfo=UTC)
END_AT = datetime(2026, 1, 3, 23, 59, 59, 999999, tzinfo=UTC)
SAMPLE_GRANULARITY = TickGranularity.TICK

data_source = AthenaDataSource.from_env().with_filters(
    TickGranularityFilter.of(SAMPLE_GRANULARITY),
)

In [3]:
parameters = SnowballStrategy.normalize_parameters(
    StrategyParameters.of(
        cycle={"hedging_enabled": False, "take_profit_pips": "20"},
        sizing={"base_units": "1000"},
        grid={
            "max_retracements_per_layer": 3,
            "max_layers": 2,
            "refill": {"enabled": True, "max_reusable_retracement": 1},
        },
        counter={
            "interval": {
                "mode": "constant",
                "head_pips": "10",
                "tail_pips": "10",
                "flat_steps": 0,
            }
        },
        stop_loss={"enabled": True, "mode": "auto"},
        rebuild={"enabled": True},
        protection={"emergency_enabled": False, "shrink_enabled": False},
    )
)

definition = BacktestTaskDefinition(
    name=f"{INSTRUMENT} Snowball Athena backtest",
    instrument=INSTRUMENT,
    start_at=START_AT,
    end_at=END_AT,
    parameters=parameters,
)

strategy = SnowballStrategy(
    name="snowball",
    parameters=definition.parameters,
)

In [4]:
recorder = RecordingEventHandler()
event_bus = EventBus(handlers=[recorder])
with TaskManager(repository=InMemoryTaskRepository(), event_bus=event_bus) as manager:
    run = manager.start_backtest(definition, data_source=data_source, strategy=strategy)
    final_task = run.wait()

print(f"Task {final_task.id} finished with status: {final_task.status.value}")
print(f"Events recorded: {len(recorder.events)}")

Task 019f4d78-1f72-7571-a676-72ceb8f22b70 finished with status: completed
Events recorded: 149


In [5]:
events = [event for event in recorder.events if isinstance(event, StrategyEvent)]


def meta(event: StrategyEvent, key: str):
    return event.metadata.get(key)


events_df = pd.DataFrame(
    {
        "timestamp": event.timestamp,
        "action": event.action.value,
        "display_id": event.display_id,
        "side": event.side.value if event.side else None,
        "units": float(event.units) if event.units is not None else None,
        "price": str(event.price) if event.price is not None else None,
        "decision": event.reason.code.value,
        "rule": event.reason.rule_id,
        "snowball_event": meta(event, "snowball_event"),
        "cycle_id": meta(event, "cycle_id"),
        "direction": meta(event, "direction"),
        "entry_role": meta(event, "entry_role"),
        "layer_number": meta(event, "layer_number"),
        "slot_number": meta(event, "slot_number"),
        "build_number": meta(event, "build_number"),
        "close_reason": meta(event, "close_reason"),
        "is_rebuild": meta(event, "is_rebuild"),
        "planned_entry_price": meta(event, "planned_entry_price"),
        "filled_entry_price": meta(event, "filled_entry_price"),
        "planned_take_profit_price": meta(event, "planned_take_profit_price"),
        "filled_take_profit_price": meta(event, "filled_take_profit_price"),
        "planned_stop_loss_price": meta(event, "planned_stop_loss_price"),
        "filled_stop_loss_price": meta(event, "filled_stop_loss_price"),
        "planned_rebuild_price": meta(event, "planned_rebuild_price"),
        "filled_rebuild_price": meta(event, "filled_rebuild_price"),
        "realized_pl": meta(event, "realized_pl"),
    }
    for event in events
)
events_df

,timestamp,action,display_id,side,units,price,decision,rule,snowball_event,cycle_id,...,is_rebuild,planned_entry_price,filled_entry_price,planned_take_profit_price,filled_take_profit_price,planned_stop_loss_price,filled_stop_loss_price,planned_rebuild_price,filled_rebuild_price,realized_pl
0,2026-01-01 21:12:58+00:00,open_trade,L1R0B1,buy,1000.0,156.71 JPY,entry_signal,snowball.open,open,1,...,False,156.71 JPY,156.71 JPY,156.91 JPY,NaN,156.61 JPY,NaN,NaN,NaN,NaN
1,2026-01-01 22:34:54+00:00,close_trade,L1R0B1,sell,1000.0,156.49 JPY,exit_signal,snowball.close.stop_loss,close,1,...,False,156.71 JPY,156.71 JPY,156.91 JPY,NaN,156.61 JPY,156.49 JPY,156.61 JPY,NaN,-218.00 JPY
2,2026-01-01 22:34:55+00:00,open_trade,L1R0B2,buy,1000.0,156.61 JPY,entry_signal,snowball.open.rebuild,open,1,...,True,156.61 JPY,156.61 JPY,156.81 JPY,NaN,156.51 JPY,NaN,156.61 JPY,156.61 JPY,NaN
3,2026-01-01 22:35:24+00:00,close_trade,L1R0B2,sell,1000.0,156.48 JPY,exit_signal,snowball.close.stop_loss,close,1,...,False,156.61 JPY,156.61 JPY,156.81 JPY,NaN,156.51 JPY,156.48 JPY,156.51 JPY,NaN,-128.00 JPY
4,2026-01-01 22:35:28+00:00,open_trade,L1R0B3,buy,1000.0,156.51 JPY,entry_signal,snowball.open.rebuild,open,1,...,True,156.51 JPY,156.51 JPY,156.71 JPY,NaN,156.41 JPY,NaN,156.51 JPY,156.51 JPY,NaN
5,2026-01-01 22:35:33+00:00,close_trade,L1R0B3,sell,1000.0,156.75 JPY,exit_signal,snowball.close.take_profit,close,1,...,False,156.51 JPY,156.51 JPY,156.71 JPY,156.75 JPY,156.41 JPY,NaN,NaN,NaN,242.00 JPY
6,2026-01-01 22:35:38+00:00,open_trade,L1R0B1,buy,1000.0,156.76 JPY,entry_signal,snowball.open,open,2,...,False,156.76 JPY,156.76 JPY,156.96 JPY,NaN,156.66 JPY,NaN,NaN,NaN,NaN
7,2026-01-01 22:35:39+00:00,close_trade,L1R0B1,sell,1000.0,156.65 JPY,exit_signal,snowball.close.stop_loss,close,2,...,False,156.76 JPY,156.76 JPY,156.96 JPY,NaN,156.66 JPY,156.65 JPY,156.66 JPY,NaN,-108.00 JPY
8,2026-01-01 22:35:43+00:00,open_trade,L1R0B2,buy,1000.0,156.66 JPY,entry_signal,snowball.open.rebuild,open,2,...,True,156.66 JPY,156.66 JPY,156.86 JPY,NaN,156.56 JPY,NaN,156.66 JPY,156.66 JPY,NaN
9,2026-01-01 22:36:09+00:00,close_trade,L1R0B2,sell,1000.0,156.41 JPY,exit_signal,snowball.close.stop_loss,close,2,...,False,156.66 JPY,156.66 JPY,156.86 JPY,NaN,156.56 JPY,156.41 JPY,156.56 JPY,NaN,-246.00 JPY
